## PolyWorld: Part 1 (Prediction) and Part 2 (Fine-Tuning)

This notebook is structured in two main parts:
1.  **Part 1: Prediction with Pre-trained Model**: Apply the original PolyWorld model to the **test dataset** to establish a baseline performance.
2.  **Part 2: Fine-Tuning the Model**: Train the model on your custom **training dataset**.


In [1]:
<VSCode.Cell id="#VSC-cd53b11c" language="python">
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import numpy as np
from PIL import Image, ImageDraw
from pathlib import Path
import sys
import json
from tqdm.notebook import tqdm

# To parse the WKT strings in your labels
from shapely import wkt as shapely_wkt

# Add the cloned repository to Python's path
sys.path.insert(0, '/media/gisense/xihan/250812_tamu_cybertraining_team4/PolyWorld')

# Import the three main components of the PolyWorld pipeline
from models.backbone import R2U_Net, NonMaxSuppression, DetectionBranch
from models.matching import OptimalMatching

print("✅ Imports and paths set up.")

✅ Imports and paths set up for training.


In [2]:
class BuildingTrainDataset(Dataset):
    def __init__(self, images_dir, labels_dir, image_size=(512, 512)):
        self.image_paths = sorted([p for p in Path(images_dir).glob("*_pre_disaster.png")])
        self.labels_dir = Path(labels_dir)
        self.image_size = image_size
        print(f"Found {len(self.image_paths)} images for training.")

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        
        # --- Load and process image ---
        img = Image.open(img_path).convert('RGB').resize(self.image_size)
        img_tensor = torch.from_numpy(np.array(img)).permute(2, 0, 1).float() / 255.0

        # --- Load label and create target grid ---
        label_path = self.labels_dir / f"{img_path.stem}.json"
        # The target grid must match the model's output size, which is the image_size
        target_grid = torch.zeros(self.image_size, dtype=torch.float32)

        if label_path.exists():
            with open(label_path) as f:
                annotations = json.load(f)
            
            # Get the original image size from metadata for scaling
            meta = annotations.get("metadata", {})
            orig_w = meta.get("width", self.image_size[0])
            orig_h = meta.get("height", self.image_size[1])

            # Get polygons from the 'xy' features
            polygons = annotations["features"]["xy"]
            for feat in polygons:
                try:
                    poly = shapely_wkt.loads(feat['wkt'])
                    for x, y in poly.exterior.coords:
                        # Scale coordinates to the new grid size (which is now image_size)
                        grid_x = int((x / orig_w) * self.image_size[1])
                        grid_y = int((y / orig_h) * self.image_size[0])
                        
                        # Clamp values to be within grid bounds
                        grid_x = max(0, min(grid_x, self.image_size[1] - 1))
                        grid_y = max(0, min(grid_y, self.image_size[0] - 1))
                        
                        # Mark this vertex in the target grid
                        target_grid[grid_y, grid_x] = 1.0
                except Exception:
                    continue # Skip invalid polygons

        return {
            'image': img_tensor,
            'target_grid': target_grid.unsqueeze(0) # Add channel dimension
        }

print("✅ Custom Training Dataset defined.")

✅ Custom Training Dataset defined.


In [ ]:
# --- Setup paths and device ---
base_repo_dir = Path("/media/gisense/xihan/250812_tamu_cybertraining_team4/PolyWorld")
pretrained_weights_dir = base_repo_dir / "trained_weights"
finetuned_output_dir = Path("/media/data/building_instance_tamu/PolyWorld/finetuned_weights")
finetuned_output_dir.mkdir(parents=True, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# --- Load model components ---
backbone = R2U_Net().to(device)
head = DetectionBranch().to(device)

# --- Load pretrained weights ---
backbone.load_state_dict(torch.load(pretrained_weights_dir / "polyworld_backbone", map_location=device))
head.load_state_dict(torch.load(pretrained_weights_dir / "polyworld_seg_head", map_location=device))

# --- Set models to training mode ---
backbone.train()
head.train()

# --- Define Optimizer and Loss Function ---
params = list(backbone.parameters()) + list(head.parameters())
optimizer = optim.Adam(params, lr=1e-5) # Small learning rate for fine-tuning
criterion = nn.BCEWithLogitsLoss() # Best for binary grid prediction

print("✅ Models loaded and training parameters are set.")

Using device: cuda
✅ Models loaded and training parameters are set.
✅ Models loaded and training parameters are set.


In [4]:
# --- Define data paths ---
# NOTE: For a real scenario, you should split your data into train/validation sets.
# Here, we use the 'test' set for demonstration.
train_images_dir = "/media/data/building_instance_tamu/test/images"
train_labels_dir = "/media/data/building_instance_tamu/test/labels"

# --- Set up the dataloader ---
train_dataset = BuildingTrainDataset(images_dir=train_images_dir, labels_dir=train_labels_dir)
train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True, num_workers=4)

# --- Training Loop ---
num_epochs = 2 # Adjust as needed
print(f"\nStarting fine-tuning for {num_epochs} epochs...")

for epoch in range(num_epochs):
    epoch_loss = 0.0
    
    # Wrap loader with tqdm for a progress bar
    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}")
    
    for batch in progress_bar:
        images = batch['image'].to(device)
        target_grids = batch['target_grid'].to(device)

        # Zero the gradients
        optimizer.zero_grad()

        # --- Forward pass ---
        features = backbone(images)
        predicted_grids = head(features)

        # --- Calculate loss ---
        loss = criterion(predicted_grids, target_grids)

        # --- Backward pass and optimization ---
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()
        
        # Update progress bar description
        progress_bar.set_postfix(loss=loss.item())

    avg_epoch_loss = epoch_loss / len(train_loader)
    print(f"Epoch {epoch+1} finished. Average Loss: {avg_epoch_loss:.6f}")

print("\n✅ Fine-tuning complete.")

Found 933 images for training.

Starting fine-tuning for 2 epochs...


Epoch 1/2:   0%|          | 0/234 [00:00<?, ?it/s]

KeyboardInterrupt: 